# NexusBrain: Enterprise Knowledge Graph Explorer
### FalkorDB Graph Hacks (Track 03: Company Brain & Track 02: Agent Memory)

This interactive notebook explores the massive synthetic enterprise dataset generated for **"7bit Cloud Technologies"**.
We inspect:
1. **Graph Scale & Node/Edge Distribution**
2. **Cross-Silo Connections & Degree Centrality**
3. **Verification of the 5 Golden Demo Scenarios** (Shadow Ownership, Temporal Decision Flip, 4-Hop Blast Radius, Cross-Agent Incident Memory, and Undocumented API Breaking Change)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Set base paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Raw Path: {DATA_RAW}")

# Load unified graph bundle
bundle_path = DATA_RAW / "graph_bundle.json"
with open(bundle_path, "r", encoding="utf-8") as f:
    bundle = json.load(f)

print(f"Successfully loaded graph_bundle.json ({bundle_path.stat().st_size:,} bytes)")

## 1. Node & Edge Inventory Breakdown

In [ ]:
node_counts = {
    "Teams": len(bundle["teams"]),
    "People": len(bundle["people"]),
    "Systems": len(bundle["systems"]),
    "Tickets": len(bundle["tickets"]),
    "PullRequests": len(bundle["pull_requests"]),
    "SlackThreads": len(bundle["slack_threads"]),
    "Documents": len(bundle["documents"]),
    "Decisions": len(bundle["decisions"]),
}

df_nodes = pd.DataFrame(list(node_counts.items()), columns=["Node Type", "Count"])
total_nodes = df_nodes["Count"].sum()
print(f"=== TOTAL NODES: {total_nodes:,} ===\n")
display(df_nodes)

# Edge distribution
df_edges = pd.DataFrame(bundle["edges"])
print(f"\n=== TOTAL EDGES: {len(df_edges):,} ===\n")
rel_counts = df_edges["relation_type"].value_counts().reset_index()
rel_counts.columns = ["Relation Type", "Count"]
display(rel_counts)

## 2. Inspecting the 5 Golden Demo Scenarios

In [ ]:
print("=================================================================")
print("SCENARIO 1: Shadow Ownership & The Hidden SME")
print("=================================================================")
df_tickets = pd.DataFrame(bundle["tickets"])
df_prs = pd.DataFrame(bundle["pull_requests"])

# Compare Liam Chen (person-04, official lead) vs Elena Rostova (person-19, SRE)
for pid, name in [("person-04", "Liam Chen (Lead)"), ("person-19", "Elena Rostova (SRE)")]:
    sev1_count = len(df_tickets[(df_tickets["assignee_id"] == pid) & (df_tickets["affected_system_id"] == "auth-service") & (df_tickets["severity"] == "SEV-1")])
    pr_count = len(df_prs[(df_prs["author_id"] == pid) & (df_prs["system_id"] == "auth-service")])
    print(f"{name} [{pid}]:")
    print(f"  - SEV-1 Tickets Resolved on auth-service: {sev1_count}")
    print(f"  - PRs Authored on auth-service: {pr_count}\n")

print("Insight: While Liam Chen is officially assigned, Elena Rostova handles all critical outages and core patches!")

In [ ]:
print("=================================================================")
print("SCENARIO 2: The Temporal Decision Flip (Knowledge Evolution)")
print("=================================================================")
df_decisions = pd.DataFrame(bundle["decisions"])
adr_051 = df_decisions[df_decisions["id"] == "ADR-051"].iloc[0]
adr_018 = df_decisions[df_decisions["id"] == "ADR-018"].iloc[0]

print(f"Old Decision: {adr_018['id']} - {adr_018['title']}")
print(f"  Status: {adr_018['status']} | Decided At: {adr_018['decided_at']}")
print(f"\nNew Decision: {adr_051['id']} - {adr_051['title']}")
print(f"  Status: {adr_051['status']} | Supersedes: {adr_051['supersedes_decision_id']}")
print(f"  Triggered By Incident: {adr_051['source_ticket_id']} | Discussed in: {adr_051['source_slack_id']}")

# Check contradiction
contradicting_edges = df_edges[(df_edges["relation_type"] == "CONTRADICTS") & (df_edges["target_id"] == "ADR-051")]
print(f"\nContradicting Documents Found:")
for _, row in contradicting_edges.iterrows():
    print(f"  - Document {row['source_id']} CONTRADICTS {row['target_id']} (Reason: {row['properties'].get('reason')})")

In [ ]:
print("=================================================================")
print("SCENARIO 3: 4-Hop Cascading Blast Radius Across Teams")
print("=================================================================")
dep_edges = df_edges[df_edges["relation_type"] == "DEPENDS_ON"]
G_deps = nx.DiGraph()
for _, row in dep_edges.iterrows():
    G_deps.add_edge(row["source_id"], row["target_id"])

chain_nodes = ["stripe-webhook-worker", "checkout-orchestrator", "auth-token-validator", "redis-session-store"]
print("Verifying Path Dependency:")
for i in range(len(chain_nodes) - 1):
    src, dst = chain_nodes[i], chain_nodes[i+1]
    has_edge = G_deps.has_edge(src, dst)
    print(f"  {src} -> DEPENDS_ON -> {dst} : {'[OK]' if has_edge else '[MISSING]'}")

print(f"\nCatalyst Incident INC-119:")
inc_119 = df_tickets[df_tickets["id"] == "INC-119"].iloc[0]
print(f"  Title: {inc_119['title']}")
print(f"  Affected System: {inc_119['affected_system_id']} (Payment Team)")
print(f"  Root Cause System: {inc_119['root_cause_system_id']} (Core Platform Team)")

In [ ]:
print("=================================================================")
print("SCENARIO 4: Cross-Agent Incident Memory (INC-045 & INC-249)")
print("=================================================================")
guide_edges = df_edges[df_edges["relation_type"] == "GUIDES_RESOLUTION_OF"]
print("Runbook Connections (GUIDES_RESOLUTION_OF):")
display(guide_edges[["source_id", "target_id", "relation_type", "properties"]])

print("\n=================================================================")
print("SCENARIO 5: Undocumented API Breaking Change (PR-398 -> INC-250)")
print("=================================================================")
pr_398 = df_prs[df_prs["id"] == "PR-398"].iloc[0]
inc_250 = df_tickets[df_tickets["id"] == "INC-250"].iloc[0]
print(f"PR-398 Title: {pr_398['title']}")
print(f"  Target System: {pr_398['system_id']}")
print(f"  Files Changed: {pr_398['files_changed']}")
print(f"\nIncident INC-250 (Triggered on downstream service):")
print(f"  Title: {inc_250['title']}")
print(f"  Affected System: {inc_250['affected_system_id']}")
print(f"  Root Cause System: {inc_250['root_cause_system_id']}")

## 3. Microservice Dependency Topology & Critical Blast Radius

In [ ]:
# Top microservices by downstream dependency (In-Degree)
in_degrees = dict(G_deps.in_degree())
top_critical_systems = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:8]

df_critical = pd.DataFrame(top_critical_systems, columns=["System ID", "Downstream Dependents (Blast Radius)"])
print("Top Tier-0/1 Services with Highest Downstream Blast Radius:")
display(df_critical)